# S2Gaussian Single-Scene Kaggle Pipeline

This notebook trains one scene at a time, saves scene-specific outputs independently, and leaves the final merge step for later.

Use one scene per run. Do not try to couple the 8 scenes into a single multi-account workflow.

In [1]:
print('hell0')

hell0


In [ ]:
from pathlib import Path
import os
import subprocess
import torch

# ==================== 1. Setup paths ====================
ROOT = Path('/kaggle/working')
# Read-only input locations (adjust to your dataset names)
INPUT_REPO = Path('/kaggle/input/datasets/kirankumarp05/kkk-ee-mcv/S2Gaussian')
DATA_ROOT = Path('/kaggle/input/datasets/kirankumarp05/ee5178-project/data')

# Writable working locations
REPO = ROOT / 'S2Gaussian'
OUTPUT_ROOT = ROOT / 'runs_s2'
LOG_ROOT = ROOT / 'logs'
REALESRGAN_DIR = ROOT / 'Real-ESRGAN'
REALESRGAN_WEIGHTS = REALESRGAN_DIR / 'RealESRGAN_x4plus.pth'

LOG_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd: str):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=True)

# ==================== 2. Copy S2Gaussian to writable area ====================
if INPUT_REPO.exists() and not REPO.exists():
    print("Copying S2Gaussian to working directory...")
    run(f"cp -r {INPUT_REPO} {REPO}")
elif not REPO.exists():
    raise RuntimeError("S2Gaussian not found in input dataset.")

# Ensure Real-ESRGAN weights are available at the path expected by train_s2gs.py.
if not REALESRGAN_WEIGHTS.exists():
    print(f"Searching for Real-ESRGAN weights to copy into {REALESRGAN_WEIGHTS}")
    found_weights = None
    for search_root in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if search_root.exists():
            for candidate in search_root.rglob('RealESRGAN_x4plus.pth'):
                found_weights = candidate
                break
        if found_weights is not None:
            break
    if found_weights is None:
        raise FileNotFoundError(
            "RealESRGAN_x4plus.pth not found under /kaggle/input or /kaggle/working. "
            "Place the weight file in a Kaggle dataset or add it to the working directory."
        )
    REALESRGAN_DIR.mkdir(parents=True, exist_ok=True)
    run(f"cp {found_weights} {REALESRGAN_WEIGHTS}")
    print(f"Copied Real-ESRGAN weights from {found_weights} to {REALESRGAN_WEIGHTS}")
else:
    print(f"Using Real-ESRGAN weights at {REALESRGAN_WEIGHTS}")

# The actual code is inside 'fsgs' subfolder (adjust if different)
REPO_CODE = REPO / 'fsgs' if (REPO / 'fsgs').exists() else REPO

print(f"Using S2Gaussian code from: {REPO_CODE}")

# ==================== 3. Install system build tools ====================
# The r2u apt warning is benign on Kaggle and can be ignored.
run("apt-get update -qq")
run("apt-get install -y -qq build-essential ninja-build")

# ==================== 4. Install Python dependencies ====================
req_file = REPO_CODE / 'requirements.txt'
if req_file.exists():
    run(f"pip install -r {req_file}")
else:
    run("pip install torch torchvision basicsr realesrgan open3d timm plyfile opencv-python tqdm")

# ==================== 5. Build diff-gaussian-rasterization ====================
dgr_path = REPO_CODE / 'submodules' / 'diff-gaussian-rasterization-confidence'
if not dgr_path.exists():
    dgr_path = REPO_CODE / 'submodules' / 'diff-gaussian-rasterization'
if dgr_path.exists():
    print("Building diff-gaussian-rasterization...")
    run(f"pip install --no-build-isolation {dgr_path}")
else:
    print("Warning: diff-gaussian-rasterization submodule not found.")

# ==================== 6. Build simple-knn ====================
knn_path = REPO_CODE / 'submodules' / 'simple-knn'

if knn_path.exists():
    print("Building simple-knn from S2Gaussian submodule...")
    os.environ["CUDA_HOME"] = "/usr/local/cuda"
    os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"  # Tesla T4
    os.environ["CUDACXX"] = "/usr/local/cuda/bin/nvcc"
    os.environ["CC"] = "gcc"
    os.environ["CXX"] = "g++"

    # Hotfix for Kaggle/Python3.12 build: FLT_MAX is used but cfloat is not included.
    simple_knn_cu = knn_path / 'simple_knn.cu'
    cu_text = simple_knn_cu.read_text()
    if '#include <cfloat>' not in cu_text and '#include <float.h>' not in cu_text:
        lines = cu_text.splitlines()
        insert_at = 0
        for i, line in enumerate(lines):
            if line.strip().startswith('#include'):
                insert_at = i + 1
        lines.insert(insert_at, '#include <cfloat>')
        simple_knn_cu.write_text('\n'.join(lines) + '\n')
        print('Patched simple_knn.cu with #include <cfloat>')

    run(f"cd {knn_path} && rm -rf build dist *.egg-info")
    run(f"pip install --no-build-isolation -v {knn_path}")
else:
    print("simple-knn submodule missing; cloning gaussian-splatting recursively for fallback...")
    run("rm -rf /tmp/gaussian-splatting")
    run("git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git /tmp/gaussian-splatting")

    # Apply same hotfix to fallback copy before build.
    fallback_cu = Path('/tmp/gaussian-splatting/submodules/simple-knn/simple_knn.cu')
    if fallback_cu.exists():
        fb_text = fallback_cu.read_text()
        if '#include <cfloat>' not in fb_text and '#include <float.h>' not in fb_text:
            fb_lines = fb_text.splitlines()
            insert_at = 0
            for i, line in enumerate(fb_lines):
                if line.strip().startswith('#include'):
                    insert_at = i + 1
            fb_lines.insert(insert_at, '#include <cfloat>')
            fallback_cu.write_text('\n'.join(fb_lines) + '\n')
            print('Patched fallback simple_knn.cu with #include <cfloat>')

    run("pip install --no-build-isolation -v /tmp/gaussian-splatting/submodules/simple-knn")

# ==================== 7. Verify builds ====================
import sys
sys.path.insert(0, str(REPO_CODE))

try:
    import diff_gaussian_rasterization
    print("✅ diff_gaussian_rasterization OK")
except ImportError as e:
    print(f"❌ diff_gaussian_rasterization import failed: {e}")

# IMPORTANT: S2Gaussian needs simple_knn._C (not just simple_knn package import).
try:
    from simple_knn._C import distCUDA2
    print("✅ simple_knn._C OK")
except Exception as e:
    print(f"❌ simple_knn._C import failed: {e}")
    raise RuntimeError("simple_knn build failed. Stop here and inspect build logs above.")

# ==================== 8. Final environment summary ====================
print("\n=== Environment Ready ===")
print(f"Data root: {DATA_ROOT} (exists: {DATA_ROOT.exists()})")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Log root: {LOG_ROOT}")
print(f"Available GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU 0: {torch.cuda.get_device_name(0)}")
    if torch.cuda.device_count() > 1:
        print(f"GPU 1: {torch.cuda.get_device_name(1)}")

Copying S2Gaussian to working directory...
Using S2Gaussian code from: /kaggle/working/S2Gaussian/fsgs
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package ninja-build.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../ninja-build_1.10.1-1_amd64.deb ...
Unpacking ninja-build (1.10.1-1) ...
Setting up ninja-build (1.10.1-1) ...
Processing triggers for man-db (2.10.2-1) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 13.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... don

In [ ]:
# Manual rebuild fallback (run only if Cell 3 still fails at simple_knn._C import).
from pathlib import Path
import subprocess

def run(cmd: str):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=True)

knn_path = Path('/kaggle/working/S2Gaussian/fsgs/submodules/simple-knn')
assert knn_path.exists(), f"simple-knn path missing: {knn_path}"

# Re-apply include fix safely.
cu_file = knn_path / 'simple_knn.cu'
txt = cu_file.read_text()
if '#include <cfloat>' not in txt and '#include <float.h>' not in txt:
    lines = txt.splitlines()
    insert_at = 0
    for i, line in enumerate(lines):
        if line.strip().startswith('#include'):
            insert_at = i + 1
    lines.insert(insert_at, '#include <cfloat>')
    cu_file.write_text('\n'.join(lines) + '\n')
    print('Patched simple_knn.cu with #include <cfloat>')

run(f"cd {knn_path} && rm -rf build dist *.egg-info")
run(f"pip install --no-build-isolation -v {knn_path}")

from simple_knn._C import distCUDA2
print('✅ simple_knn._C import succeeded after rebuild.')

In [ ]:
# Change this for each run
ACTIVE_SCENE = 'bike'

import shutil
import sys
from pathlib import Path

# Patch basicsr for torchvision>=0.25 compatibility before Stage 2 imports it.
# Make this idempotent so rerunning Cell 5 never corrupts the file.
basicsr_degradations = Path('/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py')
if basicsr_degradations.exists():
    txt = basicsr_degradations.read_text()
    lines = txt.splitlines()

    # 1) Normalize any rgb_to_grayscale import to one top-level, non-indented import.
    for i, line in enumerate(lines):
        if 'rgb_to_grayscale' in line and 'torchvision.transforms' in line:
            lines[i] = 'from torchvision.transforms.functional import rgb_to_grayscale'

    # 2) Remove stray try/except wrappers only around the normalized import line.
    remove_idx = set()
    for i, line in enumerate(lines):
        if line.strip() == 'from torchvision.transforms.functional import rgb_to_grayscale':
            for j in (i - 1, i - 2, i + 1, i + 2):
                if 0 <= j < len(lines) and lines[j].strip() in {'try:', 'except ModuleNotFoundError:'}:
                    remove_idx.add(j)

    cleaned_lines = [line for idx, line in enumerate(lines) if idx not in remove_idx]
    basicsr_degradations.write_text('\n'.join(cleaned_lines) + '\n')
    print(f"Patched basicsr compatibility: {basicsr_degradations}")
else:
    print(f"Warning: basicsr file not found at {basicsr_degradations}")

# Hotfix: make COLMAP image loading robust when rgb_mapping length differs from camera count.
dataset_readers_path = REPO_CODE / 'scene' / 'dataset_readers.py'
if dataset_readers_path.exists():
    txt = dataset_readers_path.read_text()

    # Patch 1: robust image path lookup instead of rgb_mapping[idx] only.
    old_block = """        image_path = os.path.join(images_folder, os.path.basename(extr.name))
        image_name = os.path.basename(image_path).split(\".\")[0]
        rgb_path = rgb_mapping[idx]   # os.path.join(images_folder, rgb_mapping[idx])
        rgb_name = os.path.basename(rgb_path).split(\".\")[0]
        image = Image.open(rgb_path)
"""
    new_block = """        image_path = os.path.join(images_folder, os.path.basename(extr.name))
        image_name = os.path.basename(image_path).split(\".\")[0]

        # Prefer COLMAP camera image name. Fallback to rgb_mapping safely.
        rgb_path = image_path
        if not os.path.exists(rgb_path):
            if idx < len(rgb_mapping):
                rgb_path = rgb_mapping[idx]
            else:
                base = os.path.basename(extr.name)
                matched = [p for p in rgb_mapping if os.path.basename(p) == base]
                if len(matched) > 0:
                    rgb_path = matched[0]
                else:
                    raise FileNotFoundError(
                        f\"Could not find image for camera '{extr.name}' in {images_folder}\"
                    )

        image = Image.open(rgb_path)
"""
    if old_block in txt:
        txt = txt.replace(old_block, new_block)
        print("Patched COLMAP image lookup block")
    else:
        print("COLMAP image lookup patch not needed (already patched or source changed).")

    # Patch 2: use first-entry LLFF holdout (1st, 9th, 17th, ...) on sorted images.
    old_split_a = """    if eval:
        train_cam_infos = [c for idx, c in enumerate(cam_infos) if idx % llffhold != 0]
        test_cam_infos = [c for idx, c in enumerate(cam_infos) if idx % llffhold == 0]
    else:
        train_cam_infos = cam_infos
        test_cam_infos = []
"""
    old_split_b = """    if eval:
        # Baseline-compatible LLFF holdout: 8th, 16th, ... (hold-1::hold)
        train_cam_infos = [c for idx, c in enumerate(cam_infos) if (idx + 1) % llffhold != 0]
        test_cam_infos = [c for idx, c in enumerate(cam_infos) if (idx + 1) % llffhold == 0]
    else:
        train_cam_infos = cam_infos
        test_cam_infos = []
"""
    new_split = """    if eval:
        # First-entry holdout on sorted images: 1st, 9th, 17th, ...
        train_cam_infos = [c for idx, c in enumerate(cam_infos) if idx % llffhold != 0]
        test_cam_infos = [c for idx, c in enumerate(cam_infos) if idx % llffhold == 0]
    else:
        train_cam_infos = cam_infos
        test_cam_infos = []
"""
    if old_split_a in txt:
        txt = txt.replace(old_split_a, new_split)
        print("Patched eval holdout split to first-entry behavior")
    elif old_split_b in txt:
        txt = txt.replace(old_split_b, new_split)
        print("Re-patched eval holdout split to first-entry behavior")
    elif "train_cam_infos = [c for idx, c in enumerate(cam_infos) if idx % llffhold != 0]" in txt and "test_cam_infos = [c for idx, c in enumerate(cam_infos) if idx % llffhold == 0]" in txt:
        print("Eval split already set to first-entry behavior")
    else:
        print("Eval split patch not needed (source changed).")

    dataset_readers_path.write_text(txt)
    print(f"Patched reader file: {dataset_readers_path}")
else:
    print(f"Warning: dataset_readers.py not found at {dataset_readers_path}")

# Resolve scene input path with fallback to processed data if sparse model is missing.
raw_scene_input = DATA_ROOT / ACTIVE_SCENE
processed_root_candidates = [
    DATA_ROOT.parent / 'data_processed',
    Path('/kaggle/input/datasets/kirankumarp05/ee5178-project/data_processed'),
    Path('/kaggle/input/datasets/kirankumarp05/ee5178-project/data_processed_llff'),
]

SCENE_INPUT_PATH = raw_scene_input
if not (raw_scene_input / 'sparse' / '0').exists():
    for pr in processed_root_candidates:
        cand = pr / ACTIVE_SCENE
        if (cand / 'sparse' / '0').exists() and (cand / 'images').exists():
            SCENE_INPUT_PATH = cand
            break

# If sparse files are in sparse/ (not sparse/0), normalize layout.
sparse_root = SCENE_INPUT_PATH / 'sparse'
sparse0 = sparse_root / '0'
if not sparse0.exists() and sparse_root.exists():
    required = ['cameras.bin', 'images.bin', 'points3D.bin']
    if all((sparse_root / n).exists() for n in required):
        sparse0.mkdir(parents=True, exist_ok=True)
        for n in required:
            shutil.copy2(sparse_root / n, sparse0 / n)
        print(f"Normalized sparse layout to {sparse0}")

# Writable cached copy for training (dataset_readers writes points3D.ply)
WORK_DATA_ROOT = ROOT / 'data_train'
SOURCE_PATH = WORK_DATA_ROOT / ACTIVE_SCENE

MODEL_PATH = OUTPUT_ROOT / ACTIVE_SCENE
LOG_FILE = LOG_ROOT / f'{ACTIVE_SCENE}_train.log'

print(f"Active scene: {ACTIVE_SCENE}")
print(f"Raw scene path: {raw_scene_input}")
print(f"Selected scene path: {SCENE_INPUT_PATH}")
print(f"Training scene path (writable): {SOURCE_PATH}")
print(f"Model path: {MODEL_PATH}")
print(f"Log file: {LOG_FILE}")

# Verify selected source structure.
assert SCENE_INPUT_PATH.exists(), f"Scene folder missing: {SCENE_INPUT_PATH}"
assert (SCENE_INPUT_PATH / 'images').exists(), f"Images folder missing in {SCENE_INPUT_PATH}"
assert (SCENE_INPUT_PATH / 'sparse' / '0').exists(), f"sparse/0 folder missing in {SCENE_INPUT_PATH}"

# Prepare writable training copy once per scene.
WORK_DATA_ROOT.mkdir(parents=True, exist_ok=True)
if not SOURCE_PATH.exists():
    print(f"Copying scene to writable cache: {SOURCE_PATH}")
    shutil.copytree(SCENE_INPUT_PATH, SOURCE_PATH, dirs_exist_ok=True)
else:
    print(f"Using existing writable cache: {SOURCE_PATH}")

# Quick writable check under sparse/0.
probe = SOURCE_PATH / 'sparse' / '0' / '.write_probe'
probe.write_text('ok')
probe.unlink()
print('Writable check passed.')

MODEL_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
# Sanity check: print the exact images that go to test split (first-entry holdout rule).
from pathlib import Path
import sys

# Ensure we can import COLMAP readers from the copied S2Gaussian code.
if str(REPO_CODE) not in sys.path:
    sys.path.insert(0, str(REPO_CODE))

from scene.colmap_loader import read_extrinsics_binary, read_extrinsics_text

images_bin = SOURCE_PATH / 'sparse' / '0' / 'images.bin'
images_txt = SOURCE_PATH / 'sparse' / '0' / 'images.txt'

if images_bin.exists():
    extr = read_extrinsics_binary(str(images_bin))
elif images_txt.exists():
    extr = read_extrinsics_text(str(images_txt))
else:
    raise FileNotFoundError(f"No COLMAP images file found under {SOURCE_PATH / 'sparse' / '0'}")

# Match dataset_readers flow: sort by image stem (string sort, no zero-padding).
sorted_camera_files = sorted(Path(v.name).name for v in extr.values())

llffhold = 8
test_files = [name for i, name in enumerate(sorted_camera_files) if i % llffhold == 0]
train_files = [name for i, name in enumerate(sorted_camera_files) if i % llffhold != 0]

print(f"Total views considered: {len(sorted_camera_files)}")
print(f"Train views: {len(train_files)}")
print(f"Test views (1st, 9th, 17th, ...): {len(test_files)}")
print('')
print('Test image files in order:')
for idx, name in enumerate(test_files, start=1):
    print(f"{idx:03d}: {name}")

In [ ]:
# Determine GPU IDs
num_gpus = torch.cuda.device_count()
sr_gpu_id = 1 if num_gpus >= 2 else 0
print(f"Number of GPUs: {num_gpus}")
print(f"SR GPU ID: {sr_gpu_id} (training will use GPU 0)")

# S2Gaussian defaults to --images images_8; override for your dataset layout.
images_subdir = 'images'
images_path = SOURCE_PATH / images_subdir
assert images_path.exists(), f"Expected image folder not found: {images_path}"
print(f"Using image subdir: {images_subdir}")
print("Eval mode: ON (leave every 8th image out for validation)")

# Build command
cmd = [
    'python', '-u', str(REPO / 'scripts' / 'train_s2gs.py'),
    '-s', str(SOURCE_PATH),
    '-m', str(MODEL_PATH),
    '--images', images_subdir,
    '--eval',
    '--sr_gpu_id', str(sr_gpu_id)
 ]

print(f"Command: {' '.join(cmd)}")
print(f"Training log will be saved to {LOG_FILE}")

# Run with live output
with open(LOG_FILE, 'w') as log_f:
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log_f.write(line)
    process.wait()

if process.returncode == 0:
    print(f"\n✅ Training completed for {ACTIVE_SCENE}")
    print(f"Output saved in {MODEL_PATH}")
else:
    print(f"\n❌ Training failed with exit code {process.returncode}")
    print(f"Check {LOG_FILE} for details.")

In [ ]:
# List generated files
if MODEL_PATH.exists():
    print(f"Contents of {MODEL_PATH}:")
    for f in MODEL_PATH.iterdir():
        size = f.stat().st_size / (1024*1024)
        print(f"  {f.name} ({size:.2f} MB)")
else:
    print("Model path not found.")